## Example 1: SMOTE oversampling for imbalanced datasets

In [ ]:
from imblearn.over_sampling import SMOTE
import numpy as np
from collections import Counter

# Example 1: SMOTE oversampling for imbalanced datasets
print("Example 1: SMOTE oversampling")

# Create imbalanced dataset
np.random.seed(42)
X_imbalanced = np.random.randn(100, 2)
y_imbalanced = np.array([0] * 80 + [1] * 20)  # 80% class 0, 20% class 1

print("Original dataset distribution:")
print(f"  Class distribution: {Counter(y_imbalanced)}")
print(f"  Dataset shape: {X_imbalanced.shape}")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_imbalanced, y_imbalanced)

print("After SMOTE resampling:")
print(f"  Class distribution: {Counter(y_resampled)}")
print(f"  Dataset shape: {X_resampled.shape}")
print(f"  SMOTE algorithm: {type(smote).__name__}")

# SMOTE adds synthetic minority samples until both classes match the majority
assert Counter(y_resampled) == {0: 80, 1: 80}
assert X_resampled.shape == (160, 2)
# The original samples come first and are unchanged
assert np.array_equal(X_resampled[:100], X_imbalanced)

## Example 2: Random undersampling for imbalanced datasets

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
import numpy as np
from collections import Counter

# Example 2: Random undersampling for imbalanced datasets
print("Example 2: Random undersampling")

# Create imbalanced dataset
np.random.seed(42)
X_imbalanced = np.random.randn(200, 3)
y_imbalanced = np.array([0] * 150 + [1] * 30 + [2] * 20)  # Multi-class imbalance

print("Original dataset distribution:")
print(f"  Class distribution: {Counter(y_imbalanced)}")
print(f"  Dataset shape: {X_imbalanced.shape}")

# Apply Random Undersampling
undersampler = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = undersampler.fit_resample(X_imbalanced, y_imbalanced)

print("After random undersampling:")
print(f"  Class distribution: {Counter(y_resampled)}")
print(f"  Dataset shape: {X_resampled.shape}")
print(f"  Undersampler: {type(undersampler).__name__}")

# Show sampling strategy
print(f"  Sampling strategy: {undersampler.sampling_strategy_}")

# Every class is cut down to the size of the smallest class (20)
assert Counter(y_resampled) == {0: 20, 1: 20, 2: 20}
assert X_resampled.shape == (60, 3)
# The kept rows are real rows of the input, selected by sample_indices_
assert np.array_equal(X_imbalanced[undersampler.sample_indices_], X_resampled)
assert dict(undersampler.sampling_strategy_) == {0: 20, 1: 20}

## Example 3: Combined SMOTE + Tomek links sampling

In [ ]:
from imblearn.combine import SMOTETomek
import numpy as np
from collections import Counter

# Example 3: Combined SMOTE + Tomek links sampling
print("Example 3: Combined SMOTE + Tomek links sampling")

# Create imbalanced dataset with some overlap
np.random.seed(42)
X_class0 = np.random.normal(0, 1, (80, 2))
X_class1 = np.random.normal(2, 1, (20, 2))  # Some overlap with class 0
X_imbalanced = np.vstack([X_class0, X_class1])
y_imbalanced = np.array([0] * 80 + [1] * 20)

print("Original dataset distribution:")
print(f"  Class distribution: {Counter(y_imbalanced)}")
print(f"  Dataset shape: {X_imbalanced.shape}")

# Apply SMOTE + Tomek links
smote_tomek = SMOTETomek(random_state=42)
X_resampled, y_resampled = smote_tomek.fit_resample(X_imbalanced, y_imbalanced)

print("After SMOTE + Tomek links:")
print(f"  Class distribution: {Counter(y_resampled)}")
print(f"  Dataset shape: {X_resampled.shape}")
print(f"  Combined method: {type(smote_tomek).__name__}")

# Show the two-step process
print(f"  SMOTE component: {type(smote_tomek.smote).__name__}")
print(f"  Tomek component: {type(smote_tomek.tomek).__name__}")

counts = Counter(y_resampled)
# SMOTE grows the minority class; Tomek links can only remove samples afterwards
assert 20 < counts[1] <= 80 and counts[0] <= 80
assert len(y_resampled) == X_resampled.shape[0]

## Example 4: Balanced Random Forest classifier

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

# Example 4: Balanced Random Forest classifier
print("Example 4: Balanced Random Forest classifier")

# Create imbalanced dataset
np.random.seed(42)
n_samples = 1000
X = np.random.randn(n_samples, 4)
y = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1])  # 90% class 0, 10% class 1

print("Dataset for classification:")
print(f"  Total samples: {len(X)}")
print(f"  Features: {X.shape[1]}")
print(f"  Class 0: {sum(y == 0)} samples")
print(f"  Class 1: {sum(y == 1)} samples")

# Split data
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Train Balanced Random Forest
balanced_rf = BalancedRandomForestClassifier(n_estimators=10, random_state=42)
balanced_rf.fit(X_train, y_train)

# Make predictions
y_pred = balanced_rf.predict(X_test)
print(f"Balanced Random Forest trained: {type(balanced_rf).__name__}")
print(f"Number of estimators: {balanced_rf.n_estimators}")

assert len(balanced_rf.estimators_) == 10
assert list(balanced_rf.classes_) == [0, 1]
assert y_pred.shape == (200,) and set(y_pred) <= {0, 1}
proba = balanced_rf.predict_proba(X_test)
assert proba.shape == (200, 2) and np.allclose(proba.sum(axis=1), 1.0)

## Example 5: Imbalanced-learn pipeline with ADASYN

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np
from collections import Counter

# Example 5: Imbalanced-learn pipeline with ADASYN
print("Example 5: Imbalanced-learn pipeline")

# Create imbalanced dataset with better distribution
np.random.seed(42)
X = np.random.randn(300, 3)
y = np.array([0] * 200 + [1] * 70 + [2] * 30)  # Multi-class imbalance

print("Original dataset:")
print(f"  Class distribution: {Counter(y)}")
print(f"  Dataset shape: {X.shape}")

# Create pipeline with ADASYN and Logistic Regression
pipeline = Pipeline([
    ('sampler', ADASYN(random_state=42)),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

print("Pipeline components:")
print(f"  Sampler: {type(pipeline['sampler']).__name__}")
print(f"  Classifier: {type(pipeline['classifier']).__name__}")

# Use stratified split to ensure all classes in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set class distribution: {Counter(y_train)}")

# Fit pipeline
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

print(f"Pipeline fitted and predictions made: {len(predictions)} samples")

assert predictions.shape == (60,) and set(predictions) <= {0, 1, 2}
assert Counter(y_train) == {0: 160, 1: 56, 2: 24}
# ADASYN oversamples each minority class towards the majority size
X_ada, y_ada = ADASYN(random_state=42).fit_resample(X_train, y_train)
ada_counts = Counter(y_ada)
print(f"After ADASYN: {ada_counts}")
assert ada_counts[0] == 160 and ada_counts[1] > 56 and ada_counts[2] > 24
# The sampler only runs during fit, so predict() keeps one prediction per input row
assert pipeline.predict(X_train).shape == (240,)